In [1]:
from ERA_Distribution_Classes_Python.Classes.ERADist import ERADist
from ERA_Distribution_Classes_Python.Classes.ERANataf import ERANataf
from ERA_Distribution_Classes_Python.Classes.FORM_HLRF import FORM_HLRF
from ERA_Distribution_Classes_Python.Classes.FORM_fmincon import FORM_fmincon
from ERA_Distribution_Classes_Python.Classes.SuS import SuS

In [2]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt 
from structure import Structure
from solver_2nd import Solver2ndOrder

## Material Properties

In [3]:
# NEW modified IPE 120 -> eta = 100% for Th.I.O.
E = 210e6 # kN/m2 
A = 1.321e-3 # m2
I = 0.2740555e-5 # m4
h = 0.12  # m
z = h/2  # m
sigma_yield = 35.5 # kN/cm2 
alpha = 1.14
M_yield = I/z * sigma_yield * 100e2 * 1.14 # kNm

In [4]:
def t_R(M_k, I=I, z=z, alpha=alpha):
    """
    Takes in the Steel Bending Strength M_k (random variable) in kN/cm2
    I in m^4
    z in m
    alpha is the plastic ratio 

    Returns the characteristic Bending Moment Resistance M_c_Rk for the given system in kNm
    """
    return I/z * M_k * 100e2 * alpha

## System Definition

In [5]:
# Nodes

# Node 1 
x_1 = 0.0
z_1 = 0.0
# Node 2
x_2 = 0.1
z_2 = -3.0
# Node 3
x_3 = 3.6
z_3 = -3.0

In [6]:
# Load Distribution Length (Lasteinzugsbreite)
e = 5 # m

In [7]:
# Wind Last Beiwert (Bereich D)
c_pe = 0.8 

In [8]:
# Dimensions
# Outer dimensions for beam 1
delta_x_1 = x_2 - x_1
delta_z_1 = z_2 - z_1

# Outer dimensions for beam 2
delta_x_2 = x_3 - x_2
delta_z_2 = z_3 - z_2

In [59]:
def t_S(l_1, l_2, e=e, c_pe=c_pe): # Structural Model Function
    """
    l_1 = vertical load in kN/m2
    l_2 = horizontal load in kN/m2
    e = Load Distribution Length (Lasteinzugsbreite)
    """
    s = Structure()
    # Stab 1 
    n1 = s.add_node(x=delta_x_1 * 0/15,z=delta_z_1 * 0/15, kind="frame", support={"u": True, "w": True, "phi": False})
    n2 = s.add_node(x=delta_x_1 * 1/15,z=delta_z_1 * 1/15, kind="frame", support={"u": False, "w": False, "phi": False})
    n3 = s.add_node(x=delta_x_1 * 2/15,z=delta_z_1 * 2/15, kind="frame", support={"u": False, "w": False, "phi": False})
    n4 = s.add_node(x=delta_x_1 * 3/15,z=delta_z_1 * 3/15, kind="frame", support={"u": False, "w": False, "phi": False})
    n5 = s.add_node(x=delta_x_1 * 4/15,z=delta_z_1 * 4/15, kind="frame", support={"u": False, "w": False, "phi": False})
    n6 = s.add_node(x=delta_x_1 * 5/15,z=delta_z_1 * 5/15, kind="frame", support={"u": False, "w": False, "phi": False})
    n7 = s.add_node(x=delta_x_1 * 6/15,z=delta_z_1 * 6/15, kind="frame", support={"u": False, "w": False, "phi": False})
    n8 = s.add_node(x=delta_x_1 * 7/15,z=delta_z_1 * 7/15, kind="frame", support={"u": False, "w": False, "phi": False})
    n9 = s.add_node(x=delta_x_1 * 8/15,z=delta_z_1 * 8/15, kind="frame", support={"u": False, "w": False, "phi": False})
    n10 = s.add_node(x=delta_x_1 * 9/15,z=delta_z_1 * 9/15, kind="frame", support={"u": False, "w": False, "phi": False})
    n11 = s.add_node(x=delta_x_1 * 10/15,z=delta_z_1 * 10/15, kind="frame", support={"u": False, "w": False, "phi": False})
    n12 = s.add_node(x=delta_x_1 * 11/15,z=delta_z_1 * 11/15, kind="frame", support={"u": False, "w": False, "phi": False})
    n13 = s.add_node(x=delta_x_1 * 12/15,z=delta_z_1 * 12/15, kind="frame", support={"u": False, "w": False, "phi": False})
    n14 = s.add_node(x=delta_x_1 * 13/15,z=delta_z_1 * 13/15, kind="frame", support={"u": False, "w": False, "phi": False})
    n15 = s.add_node(x=delta_x_1 * 14/15,z=delta_z_1 * 14/15, kind="frame", support={"u": False, "w": False, "phi": False})
    n16 = s.add_node(x=delta_x_1 * 15/15,z=delta_z_1 * 15/15, kind="frame", support={"u": False, "w": False, "phi": False})

    e1 = s.add_element(node_i=n1,node_j=n2, E=E, A=A, I=I)
    e2 = s.add_element(node_i=n2,node_j=n3, E=E, A=A, I=I)
    e3 = s.add_element(node_i=n3,node_j=n4, E=E, A=A, I=I)
    e4 = s.add_element(node_i=n4,node_j=n5, E=E, A=A, I=I)
    e5 = s.add_element(node_i=n5,node_j=n6, E=E, A=A, I=I)
    e6 = s.add_element(node_i=n6,node_j=n7, E=E, A=A, I=I)
    e7 = s.add_element(node_i=n7,node_j=n8, E=E, A=A, I=I)
    e8 = s.add_element(node_i=n8,node_j=n9, E=E, A=A, I=I)
    e9 = s.add_element(node_i=n9,node_j=n10, E=E, A=A, I=I)
    e10 = s.add_element(node_i=n10,node_j=n11, E=E, A=A, I=I)
    e11 = s.add_element(node_i=n11,node_j=n12, E=E, A=A, I=I)
    e12 = s.add_element(node_i=n12,node_j=n13, E=E, A=A, I=I)
    e13 = s.add_element(node_i=n13,node_j=n14, E=E, A=A, I=I)
    e14 = s.add_element(node_i=n14,node_j=n15, E=E, A=A, I=I)
    e15 = s.add_element(node_i=n15,node_j=n16, E=E, A=A, I=I)

    # Stab 2
    n17 = s.add_node(x=x_2 + delta_x_2 * 1/17,z=z_2 + delta_z_2 * 1/17,kind="frame", support={"u": False, "w": False, "phi": False})
    n18 = s.add_node(x=x_2 + delta_x_2 * 2/17,z=z_2 + delta_z_2 * 2/17,kind="frame", support={"u": False, "w": False, "phi": False})
    n19 = s.add_node(x=x_2 + delta_x_2 * 3/17,z=z_2 + delta_z_2 * 3/17,kind="frame", support={"u": False, "w": False, "phi": False})
    n20 = s.add_node(x=x_2 + delta_x_2 * 4/17,z=z_2 + delta_z_2 * 4/17,kind="frame", support={"u": False, "w": False, "phi": False})
    n21 = s.add_node(x=x_2 + delta_x_2 * 5/17,z=z_2 + delta_z_2 * 5/17,kind="frame", support={"u": False, "w": False, "phi": False})
    n22 = s.add_node(x=x_2 + delta_x_2 * 6/17,z=z_2 + delta_z_2 * 6/17,kind="frame", support={"u": False, "w": False, "phi": False})
    n23 = s.add_node(x=x_2 + delta_x_2 * 7/17,z=z_2 + delta_z_2 * 7/17,kind="frame", support={"u": False, "w": False, "phi": False})
    n24 = s.add_node(x=x_2 + delta_x_2 * 8/17,z=z_2 + delta_z_2 * 8/17,kind="frame", support={"u": False, "w": False, "phi": False})
    n25 = s.add_node(x=x_2 + delta_x_2 * 9/17,z=z_2 + delta_z_2 * 9/17,kind="frame", support={"u": False, "w": False, "phi": False})
    n26 = s.add_node(x=x_2 + delta_x_2 * 10/17,z=z_2 + delta_z_2 * 10/17,kind="frame", support={"u": False, "w": False, "phi": False})
    n27 = s.add_node(x=x_2 + delta_x_2 * 11/17,z=z_2 + delta_z_2 * 11/17,kind="frame", support={"u": False, "w": False, "phi": False})
    n28 = s.add_node(x=x_2 + delta_x_2 * 12/17,z=z_2 + delta_z_2 * 12/17,kind="frame", support={"u": False, "w": False, "phi": False})
    n29 = s.add_node(x=x_2 + delta_x_2 * 13/17,z=z_2 + delta_z_2 * 13/17,kind="frame", support={"u": False, "w": False, "phi": False})
    n30 = s.add_node(x=x_2 + delta_x_2 * 14/17,z=z_2 + delta_z_2 * 14/17,kind="frame", support={"u": False, "w": False, "phi": False})
    n31 = s.add_node(x=x_2 + delta_x_2 * 15/17,z=z_2 + delta_z_2 * 15/17,kind="frame", support={"u": False, "w": False, "phi": False})
    n32 = s.add_node(x=x_2 + delta_x_2 * 16/17,z=z_2 + delta_z_2 * 16/17,kind="frame", support={"u": False, "w": False, "phi": False})
    n33 = s.add_node(x=x_2 + delta_x_2 * 17/17,z=z_2 + delta_z_2 * 17/17,kind="frame", support={"u": False, "w": True, "phi": False})

    e16 = s.add_element(node_i=n16,node_j=n17, E=E, A=A, I=I)
    e17 = s.add_element(node_i=n17,node_j=n18, E=E, A=A, I=I)
    e18 = s.add_element(node_i=n18,node_j=n19, E=E, A=A, I=I)
    e19 = s.add_element(node_i=n19,node_j=n20, E=E, A=A, I=I)
    e20 = s.add_element(node_i=n20,node_j=n21, E=E, A=A, I=I)
    e21 = s.add_element(node_i=n21,node_j=n22, E=E, A=A, I=I)
    e22 = s.add_element(node_i=n22,node_j=n23, E=E, A=A, I=I)
    e23 = s.add_element(node_i=n23,node_j=n24, E=E, A=A, I=I)
    e24 = s.add_element(node_i=n24,node_j=n25, E=E, A=A, I=I)
    e25 = s.add_element(node_i=n25,node_j=n26, E=E, A=A, I=I)
    e26 = s.add_element(node_i=n26,node_j=n27, E=E, A=A, I=I)
    e27 = s.add_element(node_i=n27,node_j=n28, E=E, A=A, I=I)
    e28 = s.add_element(node_i=n28,node_j=n29, E=E, A=A, I=I)
    e29 = s.add_element(node_i=n29,node_j=n30, E=E, A=A, I=I)
    e30 = s.add_element(node_i=n30,node_j=n31, E=E, A=A, I=I)
    e31 = s.add_element(node_i=n31,node_j=n32, E=E, A=A, I=I)
    e32 = s.add_element(node_i=n32,node_j=n33, E=E, A=A, I=I)

    # Surface Load (kN/m2) -> Line Load (kN/m)
    l_1 *= e
    l_2 *= (e * c_pe)

    # Vertical Load on Beam 2
    s.add_dist_load(e16, qz=l_1, local=True)
    s.add_dist_load(e17, qz=l_1, local=True)
    s.add_dist_load(e18, qz=l_1, local=True)
    s.add_dist_load(e19, qz=l_1, local=True)
    s.add_dist_load(e20, qz=l_1, local=True)
    s.add_dist_load(e21, qz=l_1, local=True)
    s.add_dist_load(e22, qz=l_1, local=True)
    s.add_dist_load(e23, qz=l_1, local=True)
    s.add_dist_load(e24, qz=l_1, local=True)
    s.add_dist_load(e25, qz=l_1, local=True)
    s.add_dist_load(e26, qz=l_1, local=True)
    s.add_dist_load(e27, qz=l_1, local=True)
    s.add_dist_load(e28, qz=l_1, local=True)
    s.add_dist_load(e29, qz=l_1, local=True)
    s.add_dist_load(e30, qz=l_1, local=True)
    s.add_dist_load(e31, qz=l_1, local=True)
    s.add_dist_load(e32, qz=l_1, local=True) 


    # Wind Load on Beam 1 
    s.add_dist_load(e1, qz=l_2, local=True)
    s.add_dist_load(e2, qz=l_2, local=True)
    s.add_dist_load(e3, qz=l_2, local=True)
    s.add_dist_load(e4, qz=l_2, local=True)
    s.add_dist_load(e5, qz=l_2, local=True)
    s.add_dist_load(e6, qz=l_2, local=True)
    s.add_dist_load(e7, qz=l_2, local=True)
    s.add_dist_load(e8, qz=l_2, local=True)
    s.add_dist_load(e9, qz=l_2, local=True)
    s.add_dist_load(e10, qz=l_2, local=True)
    s.add_dist_load(e11, qz=l_2, local=True)
    s.add_dist_load(e12, qz=l_2, local=True)
    s.add_dist_load(e13, qz=l_2, local=True)
    s.add_dist_load(e14, qz=l_2, local=True)
    s.add_dist_load(e15, qz=l_2, local=True)


    #solver1 = Solver1stOrder()
    solver2 = Solver2ndOrder(tol=1e-6, max_iter=60)

    # print(f"l_1 = {l_1}")
    # print(f"l_2 = {l_2}")
    #res1 = solver1.solve(s)
    res2 = solver2.solve(s)

    #res1.print_summary()
    #res2.print_summary()

    M_max = res2.internal_forces(s.elements[14])["M_j"]

    return M_max

In [85]:
# Design Opt 1
print(t_S(l_1= 1.1*1.5, l_2= 0.65*1.5) * 1.0 / t_R(35.5))

# Design Opt 2
e_d = max(1.5 * t_S(l_1= 1.1, l_2= 0.65), 1.5 * t_S(l_1= 1.1, l_2= 0.65))
print(e_d * 1.0 / t_R(35.5))

1.1452427630413018
1.0964527164811924


In [11]:
# Vectorized Version of the Structural response function (Better for array handling later)
t_S_vectorized = np.vectorize(t_S, otypes=[float])

## Characteristic Values (for calibrating the probabilistic loads)

In [12]:
s_k = 1.1 # snow load on ground kN/m2
q_b = 0.65 # wind pressure kN/m2
w_k = q_b * c_pe # wind load kN/m2 with c_pe,10 = 0.8 (Area D)
m_k = 35.5 # kN/cm2 (steel yield resistance)

In [13]:
# # Quick Check: should give 20.86 kNm
# print(t_S(l_1=s_k * 1.5, l_2=q_b * 1.5))

## Distributions

In [14]:
# Snow time-invariant part
mu_Theta_1 = 0.81
cov_Theta_1 = 0.26
sig_Theta_1 = mu_Theta_1 * cov_Theta_1
Theta_L1 = ERADist('lognormal','MOM',[mu_Theta_1, sig_Theta_1])

# Snow load on ground
mu_L1 = 1.0
cov_L1 = 0.2
sig_L1 = mu_L1 * cov_L1
L1 = ERADist('gumbel','MOM',[mu_L1,sig_L1])

In [15]:
percentile_L1 = L1.icdf(0.98)
print(f"Snow 98% Percentile: {percentile_L1}")

Snow 98% Percentile: 1.5184551765313796


In [16]:
# Wind time-invariant part
mu_Theta_2 = 0.97
cov_Theta_2 = 0.26
sig_Theta_2 = mu_Theta_2 * cov_Theta_2
Theta_L2 = ERADist('lognormal','MOM',[mu_Theta_2, sig_Theta_2])

# Wind velocity pressure
mu_L2 = 1.0 
cov_L2 = 0.14
sig_L2 = mu_L2 * cov_L2
L2 = ERADist('gumbel','MOM',[mu_L2, sig_L2])

In [17]:
percentile_L2 = L2.icdf(0.98)
print(f"Wind 98% Percentile: {percentile_L2}")

Wind 98% Percentile: 1.3629186235719657


In [18]:
# Structural Response Model Uncertainty (from JCSS Probabilistic Model Code, Part 3, Table 3.9.1)
mu_Theta_S = 1.0
cov_Theta_S = 0.1
sig_Theta_S = mu_Theta_S * cov_Theta_S
Theta_S = ERADist('lognormal','MOM',[mu_Theta_S, sig_Theta_S])  # Distribution for Moments in frames

In [19]:
# Steel bending model uncertainty
mu_Theta_M = 1.15
cov_Theta_M = 0.05
sig_Theta_M = mu_Theta_M * cov_Theta_M
Theta_M = ERADist('lognormal','MOM',[mu_Theta_M, sig_Theta_M])

# Steel yielding strength
mu_M = 1.0
cov_M = 0.05
sig_M = mu_M * cov_M
M = ERADist('lognormal','MOM',[mu_M, sig_M])

In [20]:
percentile_M = M.icdf(0.05)
print(f"Steel 5% Percentile: {percentile_M}")

Steel 5% Percentile: 0.9199464756612658


## Shifting / Scaling Distributions

In [21]:
# Snow Load on Ground, shifted to characteristic value
snow_shift = s_k / percentile_L1 # ratio of target to current percentile, by which mean and std get multiplied

mu_L1_shifted = mu_L1 * snow_shift
sig_L1_shifted = sig_L1 * snow_shift
L1_shifted = ERADist('gumbel','MOM',[mu_L1_shifted, sig_L1_shifted])

print(f"""Snow Load on Ground gets shifted by {snow_shift}""")
print(f"""Old mean: {mu_L1}; New mean: {mu_L1_shifted}""")
print(f"""Old std: {sig_L1}; New std: {sig_L1_shifted}""")
print(f"""Old 98th percentile: {L1.icdf(.98)}; New 98th percentile: {L1_shifted.icdf(.98)}""")
print(f"""Old COV: {L1.std()/L1.mean()}; New COV: {L1_shifted.std()/L1_shifted.mean()}""")

Snow Load on Ground gets shifted by 0.7244204616646898
Old mean: 1.0; New mean: 0.7244204616646898
Old std: 0.2; New std: 0.14488409233293795
Old 98th percentile: 1.5184551765313796; New 98th percentile: 1.0999999999999999
Old COV: 0.19999999999999998; New COV: 0.19999999999999996


In [22]:
# Wind velocity pressure, shifted to characteristic value
wind_shift = q_b / percentile_L2

mu_L2_shifted = mu_L2 * wind_shift
sig_L2_shifted = sig_L2 * wind_shift
L2_shifted = ERADist('gumbel','MOM',[mu_L2_shifted, sig_L2_shifted])

print(f"""Wind velocity pressure gets shifted by {wind_shift}""")
print(f"""Old mean: {mu_L2}; New mean: {mu_L2_shifted}""")
print(f"""Old std: {sig_L2}; New std: {sig_L2_shifted}""")
print(f"""Old 98th percentile: {L2.icdf(.98)}; New 98th percentile: {L2_shifted.icdf(.98)}""")
print(f"""Old COV: {L2.std()/L2.mean()}; New COV: {L2_shifted.std()/L2_shifted.mean()}""")

Wind velocity pressure gets shifted by 0.4769176888173018
Old mean: 1.0; New mean: 0.4769176888173018
Old std: 0.14; New std: 0.06676847643442226
Old 98th percentile: 1.3629186235719657; New 98th percentile: 0.65
Old COV: 0.14; New COV: 0.13999999999999999


In [23]:
# Steel bending resistance, shifted to characteristic value
steel_shift = m_k / percentile_M

mu_M_shifted = mu_M * steel_shift
sig_M_shifted = sig_M * steel_shift
M_shifted = ERADist('lognormal','MOM',[mu_M_shifted, sig_M_shifted])

print(f"""Steel bending resistance gets shifted by {steel_shift}""")
print(f"""Old mean: {mu_M}; New mean: {mu_M_shifted}""")
print(f"""Old std: {sig_M}; New std: {sig_M_shifted}""")
print(f"""Old 5th percentile: {M.icdf(0.05)}; New 5th percentile: {M_shifted.icdf(0.05)}""")
print(f"""Old COV: {M.std()/M.mean()}; New COV: {M_shifted.std()/M_shifted.mean()}""")

Steel bending resistance gets shifted by 38.58920158858403
Old mean: 1.0; New mean: 38.58920158858403
Old std: 0.05; New std: 1.9294600794292016
Old 5th percentile: 0.9199464756612658; New 5th percentile: 35.5
Old COV: 0.04999999999999947; New COV: 0.04999999999999946


### Distribution Plots

In [24]:
x_plotting = np.linspace(0, 2.0, 200) # for PDF plotting only
x_resistance_plotting = np.linspace(30, 50, 200) # for PDF plotting only

In [25]:
# # FIGURE 1: LOADS Side by side
# # ================================================================================
# fig1, axes = plt.subplots(1, 2, figsize=(16, 5))

# # --- Left Plot: SNOW ---
# axes[0].plot(x_plotting, Theta_L1.pdf(x_plotting), label=fr'$\Theta_1$ (Lognormal, $\mu={Theta_L1.mean():.2f}, COV={Theta_L1.std()/Theta_L1.mean():.2f}$)')
# axes[0].plot(x_plotting, L1.pdf(x_plotting), label=fr'$Q_1$ (Gumbel, $\mu={L1.mean():.2f}, COV={L1.std()/L1.mean():.2f}$)')
# axes[0].plot(x_plotting, L1_shifted.pdf(x_plotting), label=fr'$Q_1,shifted$ (Gumbel, $\mu={L1_shifted.mean():.2f}, COV={L1_shifted.std()/L1_shifted.mean():.2f}$)', linestyle= "--", color = "orange")
# axes[0].set_title('Snow Load Components')
# axes[0].set_xlabel('$x$')
# axes[0].set_ylabel('$f(x)$')
# axes[0].grid(True, linestyle='--', alpha=0.6)
# axes[0].legend(fontsize="medium")
# axes[0].set_ylim(top=axes[0].get_ylim()[1] * 1.25)

# # --- Right Plot: WIND ---
# axes[1].plot(x_plotting, Theta_L2.pdf(x_plotting), label=fr'$\Theta_2$ (Lognormal, $\mu={Theta_L2.mean():.2f}, COV={Theta_L2.std()/Theta_L2.mean():.2f}$)')
# axes[1].plot(x_plotting, L2.pdf(x_plotting), label=fr'$Q_2$ (Gumbel, $\mu={L2.mean():.2f}, COV={L2.std()/L2.mean():.2f}$)')
# axes[1].plot(x_plotting, L2_shifted.pdf(x_plotting), label=fr'$Q_2,shifted$ (Gumbel, $\mu={L2_shifted.mean():.2f}, COV={L2_shifted.std()/L2_shifted.mean():.2f}$)', linestyle= "--", color = "orange")
# axes[1].set_title('Wind Load Components')
# axes[1].set_xlabel('$x$')
# axes[1].set_ylabel('$f(x)$')
# axes[1].grid(True, linestyle='--', alpha=0.6)
# axes[1].legend(fontsize="medium")
# axes[1].set_ylim(top=axes[1].get_ylim()[1] * 1.25)

# fig1.tight_layout()
# plt.show()


In [26]:
# # FIGURE 2: Resistance
# # ================================================================================
# fig2, axes = plt.subplots(1, 2, figsize=(16, 5))

# # --- Left Plot: Resistance Original ---
# axes[0].plot(x_plotting, Theta_M.pdf(x_plotting), label=fr'$\Theta_M$ (Lognormal, $\mu={Theta_M.mean():.2f}, COV={Theta_M.std()/Theta_M.mean():.2f}$)', color='blue')
# axes[0].plot(x_plotting, M.pdf(x_plotting), label=fr'$M$ (Lognormal, $\mu={M.mean():.2f}, COV={M.std()/M.mean():.2f}$)', color='orange')
# axes[0].set_title('Steel Yielding Strength (Resistance)')
# axes[0].set_xlabel('$x$')
# axes[0].set_ylabel('$f(x)$')
# axes[0].grid(True, linestyle='--', alpha=0.6)
# axes[0].legend(fontsize="medium")
# axes[0].set_ylim(top=axes[0].get_ylim()[1] * 1.25)

# # --- Right Plot: Resistance Shifted (without Model uncertainty) --
# axes[1].plot(x_resistance_plotting, M_shifted.pdf(x_resistance_plotting), label=fr'$M,shifted$ (Lognormal, $\mu={M_shifted.mean():.2f}, COV={M_shifted.std()/M_shifted.mean():.2f}$)', color='orange', linestyle= "--")
# axes[1].set_title('Steel Yielding Strength (Resistance)')
# axes[1].set_xlabel('$x$')
# axes[1].set_ylabel('$f(x)$')
# axes[1].grid(True, linestyle='--', alpha=0.6)
# axes[1].legend(fontsize="medium")
# axes[1].set_ylim(top=axes[1].get_ylim()[1] * 1.25)

# fig1.tight_layout()
# plt.show()

## Partial Safety Factors

In [27]:
gamma_M = 1.0  # Resistance
gamma_F1 = 1.5   # Snow Load
gamma_F2 = 1.5   # Wind Load
#psi_0 = 1.0 # 0.6     # Windload

In [28]:
# Array of marginal distributions
marginal_dist = [Theta_M, M_shifted, Theta_L1, L1_shifted, Theta_L2, L2_shifted, Theta_S]

# Correlation matrix (no correlation yet)
dimensions = len(marginal_dist)
R_xx = np.eye(dimensions)

# Construction of the Nataf Distribution
nataf = ERANataf(M=marginal_dist, Correlation=R_xx)

## Subset Simulation

### Design Option 1

In [86]:
# deterministic design action effect
e_d_opt1 = t_S(l_1=gamma_F1 * s_k, l_2=gamma_F2 * q_b) # kNm

In [ ]:
def g_opt_1_sus(x):
    """
    LSF for Design Option 1
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k) # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt1 / r_k) * x[:,0] * x[:,1]
    action_side = x[:,6] * t_S_vectorized(l_1=(x[:,2] * x[:,3]), l_2=(x[:,4] * x[:,5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [ ]:
# # %% Samples Return
# samples_return = 1
# # %% subset simulation
# N  = 10000        # Total number of samples for each level
# p0 = 0.1         # Probability of each subset, chosen adaptively

# print('\n\nSUBSET SIMULATION: ')
# [Pf_SuS_1, delta_SuS, b, Pf, b_sus, pf_sus, samplesU, samplesX, fs_iid] = SuS(N, p0, g_opt_1_sus, nataf, samples_return)



SUBSET SIMULATION: 
Evaluating performance function:	OK!

-Threshold intermediate level  0  =  36.25487466181469
	*aCS lambda = 0.7196139460768353 	*aCS sigma = 0.7196139460768353 	*aCS accrate = 0.438608305274972

-Threshold intermediate level  1  =  30.98673952112703
	*aCS lambda = 0.5213332215979294 	*aCS sigma = 0.5213332215979294 	*aCS accrate = 0.4337822671156004

-Threshold intermediate level  2  =  26.629548518332683
	*aCS lambda = 0.42538749580839924 	*aCS sigma = 0.42538749580839924 	*aCS accrate = 0.43815937149270484

-Threshold intermediate level  3  =  22.337007279626516
	*aCS lambda = 0.39767311297028074 	*aCS sigma = 0.39767311297028074 	*aCS accrate = 0.4439955106621774

-Threshold intermediate level  4  =  17.943150849147436
	*aCS lambda = 0.3423529924273592 	*aCS sigma = 0.3423529924273592 	*aCS accrate = 0.44036195286195284

-Threshold intermediate level  5  =  13.104010352371184
	*aCS lambda = 0.32110195801574204 	*aCS sigma = 0.32110195801574204 	*aCS accrate = 0

In [ ]:
# print("Subset Simulation for Design Option 1")
# print(f"P(F) = {Pf_SuS_1}")
# X = sp.stats.Normal()
# beta = - X.icdf(Pf_SuS_1)
# print(f"beta = {beta}")
# print(samplesX)

Subset Simulation for Design Option 1
P(F) = 3.725000000000002e-09
beta = 5.780452434272495
[array([[ 1.09331532, 38.69181405,  0.98592322, ...,  2.43202114,
         0.87470957,  1.43490891],
       [ 1.09331532, 38.69181405,  0.98592322, ...,  2.43202114,
         0.87470957,  1.43490891],
       [ 1.08499947, 38.92622189,  0.92662426, ...,  2.35920419,
         0.82817021,  1.46409798],
       ...,
       [ 1.13717332, 36.59523565,  0.76662127, ...,  2.49960142,
         1.05866524,  1.22411126],
       [ 1.03051228, 40.70085722,  1.22568388, ...,  3.85605452,
         0.76320491,  1.10971359],
       [ 1.03504929, 41.0795976 ,  1.20778534, ...,  3.69771995,
         0.72486431,  1.13020619]], shape=(10000, 7))]


---

### Design Option 2

In [33]:
# deterministic design action effect
argument_1 = gamma_F1 * t_S(l_1= s_k, l_2= (gamma_F2 / gamma_F1) * q_b)
argument_2 = gamma_F2 * t_S(l_1= (gamma_F1 / gamma_F2) * s_k, l_2= q_b)
e_d_opt2 = max(argument_1, argument_2) # kNm

In [34]:
def g_opt_2_sus(x):
    """
    LSF for Design Option 2
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt2 / r_k) * x[:,0] * x[:,1]
    action_side = x[:,6] * t_S_vectorized(l_1=(x[:,2] * x[:,3]), l_2=(x[:,4] * x[:,5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [ ]:
# # %% Samples Return
# samples_return = 1
# # %% subset simulation
# N  = 10000        # Total number of samples for each level
# p0 = 0.1         # Probability of each subset, chosen adaptively

# print('\n\nSUBSET SIMULATION: ')
# [Pf_SuS_2, delta_SuS, b, Pf, b_sus, pf_sus, samplesU, samplesX, fs_iid] = SuS(N, p0, g_opt_2_sus, nataf, samples_return)



SUBSET SIMULATION: 
Evaluating performance function:	OK!

-Threshold intermediate level  0  =  34.24826865608187
	*aCS lambda = 0.7681728330245818 	*aCS sigma = 0.7681728330245818 	*aCS accrate = 0.44691358024691363

-Threshold intermediate level  1  =  29.14515242676853
	*aCS lambda = 0.5308787745645454 	*aCS sigma = 0.5308787745645454 	*aCS accrate = 0.43434343434343425

-Threshold intermediate level  2  =  24.869097624608564
	*aCS lambda = 0.4378880243995184 	*aCS sigma = 0.4378880243995184 	*aCS accrate = 0.4388327721661055

-Threshold intermediate level  3  =  20.56965405093116
	*aCS lambda = 0.37144516594063826 	*aCS sigma = 0.37144516594063826 	*aCS accrate = 0.4362514029180696

-Threshold intermediate level  4  =  16.144149231117304
	*aCS lambda = 0.330381822992345 	*aCS sigma = 0.330381822992345 	*aCS accrate = 0.43782267115600454

-Threshold intermediate level  5  =  11.517173782781057
	*aCS lambda = 0.30484994349569666 	*aCS sigma = 0.30484994349569666 	*aCS accrate = 0.43

In [ ]:
# print("Subset Simulation for Design Option 2")
# print(f"P(F) = {Pf_SuS_2}")

# X = sp.stats.Normal()
# beta = - X.icdf(Pf_SuS_2)
# print(f"beta = {beta}")
# # print(samplesX)

Subset Simulation for Design Option 2
P(F) = 6.678000000000003e-09
beta = 5.68144687738586


## FORM Analysis

### Design Option 1

In [82]:
def g_opt_1(x):
    """
    LSF for Design Option 1
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k) # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt1 / r_k) * x[0] * x[1]
    action_side = x[6] * t_S_vectorized(l_1=(x[2] * x[3]), l_2=(x[4] * x[5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [81]:
# Perform FORM with HLRF
# [u_star, x_star, beta, Pf, S_F1, S_F1_T] = FORM_HLRF(g=g_opt_1, dg=[], distr=nataf, sensitivity_analysis=0, u0=0)

# Perform FORM with fmincom
[u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_1, dg=[], distr=nataf, u0=0)


*scipy.optimize.minimize() with  SLSQP  Method

  22  iterations... Reliability index =  5.644944887027754  --- Failure probability =  8.261708579233328e-09 




In [39]:
print(f"u_star = {u_star}")
print(f"x_star = {x_star}")
print(f"alpha_orig = {alpha}")
print(f"alpha_manual = {u_star / beta}")
print(f"beta = {beta}")
print(f"P(F) = {Pf}")
print(f"g(X*) = {g_opt_1(x_star)}")

u_star = [-0.90822255 -0.90822272  0.29886757  0.21841252  4.05180837  3.5262672
  1.81285599]
x_star = [ 1.0976052  36.8310505   0.84620871  0.73015388  2.6461994   0.88754001
  1.19227072]
alpha_orig = [-2.01000022e-06  9.99999908e-01 -3.49085173e-06 -1.42588596e-06
 -4.28846219e-04 -3.68483102e-06 -5.66608157e-06]
alpha_manual = [-0.15593114 -0.15593117  0.05131205  0.03749886  0.6956479   0.60541865
  0.31124608]
beta = 5.824510353292241
P(F) = 2.8640143644569173e-09
g(X*) = 1.7849826150495574e-07


### Design Option 2

In [142]:
def g_opt_2(x):
    """
    LSF for Design Option 2
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M *e_d_opt2 / r_k) * x[0] * x[1]
    action_side = x[6] * t_S_vectorized(l_1=(x[2] * x[3]), l_2=(x[4] * x[5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [143]:
# Perform FORM with HLRF
# [u_star, x_star, beta, Pf, S_F1, S_F1_T] = FORM_HLRF(g=g_opt_2, dg=[], distr=nataf, sensitivity_analysis=0, u0=1)

# Perform FORM with fmincom
[u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_2, dg=[], distr=nataf)


*scipy.optimize.minimize() with  SLSQP  Method

  24  iterations... Reliability index =  5.688956651141901  --- Failure probability =  6.390895375924368e-09 




In [43]:
print(f"u_star = {u_star}")
print(f"x_star = {x_star}")
print(f"alpha_orig = {alpha}")
print(f"alpha_manual = {u_star / beta}")
print(f"beta = {beta}")
print(f"P(F) = {Pf}")
print(f"g(X*) = {g_opt_2(x_star)}")

u_star = [-0.88756144 -0.88756135  0.30465068  0.22018404  3.97423697  3.42250662
  1.77180506]
x_star = [ 1.09873897 36.86909539  0.84746125  0.73040304  2.59421708  0.86740394
  1.18739849]
alpha_orig = [-2.02074987e-06  9.99999923e-01 -3.51892502e-06 -1.43187813e-06
 -3.93525997e-04 -3.34533326e-06 -5.57459486e-06]
alpha_manual = [-0.1560148  -0.15601479  0.05355124  0.03870376  0.69858802  0.60160533
  0.3114464 ]
beta = 5.688956651141901
P(F) = 6.390895375924368e-09
g(X*) = 3.664028014327414e-08


## Optimization of additional PSF $\gamma_{new}$

Remark: due to long simulation time with SuS, only FORM is used here

In [115]:
from scipy.optimize import minimize
from scipy.optimize import brentq

In [ ]:
beta_TRG = 5.230751 # TH1 case FORM

### Design Option (1) 

In [ ]:
def f(gamma_new):
    def g_opt_1(x):
        """
        LSF for Design Option 1
        Input variables: 
        x[0]: Theta_M = Resistance Model Uncertainty
        x[1]: M = Steel yield strength
        x[2]: Theta_L1 = Snow Load Model Uncertainty
        x[3]: L1 = Snow Load on Ground
        x[4]: Theta_L2 = Wind Load Model Uncertainty
        x[5]: L2 = Wind velocity pressure
        x[6]: Theta_S = Structural Response Model Uncertainty
        """
        
        # deterministic characteristic resistance 
        r_k = t_R(M_k=m_k) # kNm
        
        # assembly of the LSF
        resistance_side = (gamma_M * gamma_new *e_d_opt1 / r_k) * x[0] * x[1]
        action_side = x[6] * t_S_vectorized(l_1=(x[2] * x[3]), l_2=(x[4] * x[5]))
        
        # print(f"resistance = {resistance_side}")
        # print(f"action = {action_side}")
        
        return resistance_side - action_side

    [u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_1, dg=[], distr=nataf)
    return beta - beta_TRG

In [ ]:
# this is for narrowing down the lower and upper bound of where brentq should search in the input space
# for g in [0.8, 0.85 ,0.9]:
#     print(g, f(g))


*scipy.optimize.minimize() with  SLSQP  Method

  21  iterations... Reliability index =  5.131426675267906  --- Failure probability =  1.4377712152931287e-07 


0.8 -0.09932432473209385


/var/folders/lj/y31_mc2n4cj8z4jb_s7xdnqr0000gn/T/ipykernel_97946/2800514093.py:25: RuntimeWarning: invalid value encountered in scalar subtract
  return resistance_side - action_side



*scipy.optimize.minimize() with  SLSQP  Method

  29  iterations... Reliability index =  5.3193919350507075  --- Failure probability =  5.205731592316662e-08 


0.85 0.0886409350507078

*scipy.optimize.minimize() with  SLSQP  Method

  23  iterations... Reliability index =  5.4968358725292585  --- Failure probability =  1.9333313251443126e-08 


0.9 0.26608487252925883


Observation: root lies between $x = 0.8$ and $x = 0.85$


In [ ]:
# Finding the Root (Optimization Problem)
gamma_solution = brentq(f, 0.8, 0.85)


*scipy.optimize.minimize() with  SLSQP  Method

  21  iterations... Reliability index =  5.131426675267906  --- Failure probability =  1.4377712152931287e-07 




/var/folders/lj/y31_mc2n4cj8z4jb_s7xdnqr0000gn/T/ipykernel_97946/2800514093.py:25: RuntimeWarning: invalid value encountered in scalar subtract
  return resistance_side - action_side



*scipy.optimize.minimize() with  SLSQP  Method

  29  iterations... Reliability index =  5.3193919350507075  --- Failure probability =  5.205731592316662e-08 



*scipy.optimize.minimize() with  SLSQP  Method

  28  iterations... Reliability index =  5.232139867870172  --- Failure probability =  8.377944188090753e-08 



*scipy.optimize.minimize() with  SLSQP  Method

  17  iterations... Reliability index =  5.230783275239267  --- Failure probability =  8.439665253099931e-08 



*scipy.optimize.minimize() with  SLSQP  Method

  23  iterations... Reliability index =  5.230728842513118  --- Failure probability =  8.442150937459604e-08 



*scipy.optimize.minimize() with  SLSQP  Method

  19  iterations... Reliability index =  5.230732518843822  --- Failure probability =  8.44198303459132e-08 



*scipy.optimize.minimize() with  SLSQP  Method

  25  iterations... Reliability index =  5.230760883530352  --- Failure probability =  8.440687690527645e-08 



*scipy.optimize.minimize() with  

In [124]:
print(f"gamma_new = {gamma_solution}")

gamma_new = 0.8260468918830731


### Verification of $\gamma_{new}$ for Design Option (1)

In [120]:
def g_opt_1(x):
    """
    LSF for Design Option 1
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k) # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * gamma_solution * e_d_opt1 / r_k) * x[0] * x[1]
    action_side = x[6] * t_S_vectorized(l_1=(x[2] * x[3]), l_2=(x[4] * x[5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [121]:
# Perform FORM with fmincom
[u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_1, dg=[], distr=nataf, u0=0)


*scipy.optimize.minimize() with  SLSQP  Method

  24  iterations... Reliability index =  5.230736782135288  --- Failure probability =  8.44178832849455e-08 




In [123]:
print(f"Target Reliability Index: {beta_TRG}")
print(f"Optimized Reliability Index: {beta}")
print(f"Difference: {beta - beta_TRG}")

Target Reliability Index: 5.230751
Optimized Reliability Index: 5.230736782135288
Difference: -1.4217864711341122e-05


### Design Option (2) 

In [145]:
def f(gamma_new):
    def g_opt_2(x):
        """
        LSF for Design Option 2
        Input variables: 
        x[0]: Theta_M = Resistance Model Uncertainty
        x[1]: M = Steel yield strength
        x[2]: Theta_L1 = Snow Load Model Uncertainty
        x[3]: L1 = Snow Load on Ground
        x[4]: Theta_L2 = Wind Load Model Uncertainty
        x[5]: L2 = Wind velocity pressure
        x[6]: Theta_S = Structural Response Model Uncertainty
        """
        
        # deterministic characteristic resistance 
        r_k = t_R(M_k=m_k)  # kNm
        
        # assembly of the LSF
        resistance_side = (gamma_M * gamma_new * e_d_opt2 / r_k) * x[0] * x[1]
        action_side = x[6] * t_S_vectorized(l_1=(x[2] * x[3]), l_2=(x[4] * x[5]))
        
        # print(f"resistance = {resistance_side}")
        # print(f"action = {action_side}")
        
        return resistance_side - action_side

    [u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_2, dg=[], distr=nataf)
    return beta - beta_TRG

In [146]:
# # this is for narrowing down the lower and upper bound of where brentq should search in the input space
# for g in [0.7, 0.8 ,0.9, 1.0]:
#     print(g, f(g))

Observation: root lies between $x = 0.8$ and $x = 0.9$. 
Narrowed down a little more to 0.85 and 0.87

In [147]:
# Finding the Root (Optimization Problem)
gamma_solution = brentq(f, 0.85, 0.87)


*scipy.optimize.minimize() with  SLSQP  Method

  29  iterations... Reliability index =  5.184403808696989  --- Failure probability =  1.0835347705398408e-07 



*scipy.optimize.minimize() with  SLSQP  Method

  24  iterations... Reliability index =  5.256553391911387  --- Failure probability =  7.339008860797019e-08 



*scipy.optimize.minimize() with  SLSQP  Method

  24  iterations... Reliability index =  5.230951568990141  --- Failure probability =  8.431984551181257e-08 



*scipy.optimize.minimize() with  SLSQP  Method

  23  iterations... Reliability index =  5.230740253449278  --- Failure probability =  8.441629795508784e-08 



*scipy.optimize.minimize() with  SLSQP  Method

  23  iterations... Reliability index =  5.230721109373895  --- Failure probability =  8.442504130658578e-08 




/Users/johannesstumpfl/Desktop/BAU-ing/5.Semester TUM/Masterarbeit/Python DSM Analysis/core/loads.py:105: RuntimeWarning: invalid value encountered in matmul
  f_glo = T.T @ f_lok


RuntimeError: Keine Konvergenz nach 60 Iterationen. Knicklast überschritten oder tol zu streng?

In [ ]:
print(f"gamma_new = {gamma_solution}")

### Verification of $\gamma_{new}$ for Design Option (2)